# 5. GPT-2 inference with metrics


In [1]:
# Dynamicly load evaluation metrics and the model
%run -i ../src/models/evaluation.py
%run -i ../src/models/detoxGPT2.py

In [2]:
import pandas as pd
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
detoxGPT = detoxGPT2()

# Instantiate metric classes
similarity = Similarity()
toxicity = STAToxic()

In [4]:
prompt = "What a fucking stupid thing to say!"

In [5]:
# Pack the suggestions into a dataframe
df = pd.DataFrame(
    detoxGPT.get_detoxed_suggestions(prompt, device=DEVICE), columns=["suggestion"]
)

# Add empty column for each metric
metrics = ["wo", "cs", "bleu"]
df[metrics] = pd.DataFrame([[0] * len(metrics)], index=df.index, dtype=float)

# Generate toxicity report for each suggestion
toxicity_report = toxicity.toxicity_report(df["suggestion"])
toxicity_report_columns = toxicity_report.columns

for index, row in df.iterrows():
    df.loc[index, "wo"] = similarity.get_wo_score(prompt, row["suggestion"])
    df.loc[index, "cs"] = similarity.get_cosine_score(prompt, row["suggestion"])
    df.loc[index, "bleu"] = similarity.get_bleu_score(prompt, row["suggestion"])

# Concat with toxicity report
df = pd.concat([df, toxicity_report], axis=1)
df

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


,suggestion,wo,cs,bleu,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,what a terrible thing to say!,0.444444,0.972815,0.510691,0.131332,0.000517,0.007929,0.000122,0.012770,0.00113
1,What a terrible thing to say!,0.444444,0.985969,0.540665,0.131332,0.000517,0.007929,0.000122,0.012770,0.00113
2,what a nasty thing you should say!,0.272727,0.924192,0.448248,0.171508,0.002237,0.018131,0.000611,0.067049,0.00153
3,What a terrible thing to say!,0.444444,0.985969,0.540665,0.131332,0.000517,0.007929,0.000122,0.012770,0.00113
4,What a terrible thing to say!,0.444444,0.985969,0.540665,0.131332,0.000517,0.007929,0.000122,0.012770,0.00113
5,What a bad line to say!,0.300000,0.943460,0.358246,0.039578,0.001355,0.003196,0.000155,0.015327,0.00021
6,It's a terrible thing to say.,0.300000,0.889844,0.400643,0.131332,0.000517,0.007929,0.000122,0.012770,0.00113


In [6]:
# Calculate the score
metric_weights = {"wo": 0.1, "cs": 0.5, "bleu": 0.4}

# Toxicity report should be as low as possible
# Similarity metrics should be as high as possible (use weighted sum)
df["score"] = 1 / df[toxicity_report_columns].sum(axis=1)
df["score"] *= df[metrics].dot(pd.Series(metric_weights))

# Sort by score
df = df.sort_values(by=["score"], ascending=False)
df

,suggestion,wo,cs,bleu,toxic,severe_toxic,obscene,threat,insult,identity_hate,score
5,What a bad line to say!,0.300000,0.943460,0.358246,0.039578,0.001355,0.003196,0.000155,0.015327,0.00021,10.782571
1,What a terrible thing to say!,0.444444,0.985969,0.540665,0.131332,0.000517,0.007929,0.000122,0.012770,0.00113,4.900464
3,What a terrible thing to say!,0.444444,0.985969,0.540665,0.131332,0.000517,0.007929,0.000122,0.012770,0.00113,4.900464
4,What a terrible thing to say!,0.444444,0.985969,0.540665,0.131332,0.000517,0.007929,0.000122,0.012770,0.00113,4.900464
0,what a terrible thing to say!,0.444444,0.972815,0.510691,0.131332,0.000517,0.007929,0.000122,0.012770,0.00113,4.779747
6,It's a terrible thing to say.,0.300000,0.889844,0.400643,0.131332,0.000517,0.007929,0.000122,0.012770,0.00113,4.129884
2,what a nasty thing you should say!,0.272727,0.924192,0.448248,0.171508,0.002237,0.018131,0.000611,0.067049,0.00153,2.561292


In [7]:
# Print the suggestion with the highest score
suggestion = df.iloc[0]["suggestion"]
print(f"{prompt} -> {suggestion}")

What a fucking stupid thing to say! -> What a bad line to say!
